In [23]:
import os
from dotenv import load_dotenv
from google import genai

In [24]:
load_dotenv()  # Loads variables from .env

api_key = os.getenv("GEMINI_API_KEY")

In [25]:
client = genai.Client(
    api_key=api_key,
)

In [19]:
generation_config = {
    'temperature': 1,
    'max_output_tokens': 65536,
    'top_p': 0.95,
    'thinking_level': 'low',
}

def prompt_passing(prompt):
    interaction = client.interactions.create(
        model='models/gemini-3-flash-preview',
        input=prompt,
        system_instruction="""Extract specified information fields from an unstructured client enquiry and return all results as a valid JSON object conforming to the schema below. Use only information explicitly present in the enquiry. Do not make assumptions or fabricate any details not supplied. Identify any required fields (from the schema) that are missing, and list them in the `missing_information` array. If information is partially present (e.g., only a first name without a company), include what is present and note the remainder as missing. Always ensure JSON validity and field accuracy.

Persist in analysis and clarification: if you encounter ambiguity, carefully reason through what can and cannot be deduced based on the text before producing your output. Think step-by-step through the extraction for each field. Only after thoroughly considering all possible information present should you output your structured data.

Output Format:
- Return a single, well-formatted JSON object only, matching the schema exactly.
- If a field cannot be determined from the enquiry, leave it as an empty string \"\".
- Fill the \"missing_information\" array with the field names (as strings) of all fields not deducible from the enquiry.

Schema:
{
  \"client_name\": \"\",
  \"company\": \"\",
  \"industry\": \"\",
  \"requirement\": \"\",
  \"urgency\": \"\",
  \"missing_information\": []
}

Example 1  
Input:  
Hi, my name is Anna, and I need help with digital marketing for my tech startup. We're hoping to get started as soon as possible.

Expected Output:
{
  \"client_name\": \"Anna\",
  \"company\": \"\",
  \"industry\": \"tech startup\",
  \"requirement\": \"help with digital marketing\",
  \"urgency\": \"as soon as possible\",
  \"missing_information\": [\"company\"]
}

Example 2  
Input:  
Greetings, I'm from SolarPower Inc., working in the renewable energy sector. I'm looking for information on your consulting packages.

Expected Output:
{
  \"client_name\": \"\",
  \"company\": \"SolarPower Inc.\",
  \"industry\": \"renewable energy\",
  \"requirement\": \"information on consulting packages\",
  \"urgency\": \"\",
  \"missing_information\": [\"client_name\", \"urgency\"]
}

Edge Cases & Details:
- Do not invent names, companies, or industries.
- If more than one field is missing, list all in the missing_information array.
- Partial mentions (e.g. “Sarah from Finance”) should populate what is stated and missing elements identified.
- The JSON output should not be wrapped in quotes or code blocks.

Reminders:
- Strictly do not invent or assume information. Only use what’s in the input.
- Always return a strictly valid JSON object, matching the specified schema, and update missing_information accordingly.
""",
        generation_config=generation_config,
)
    return interaction

prompt = "Hi, I'm Jason from GreenTech Energy. We're looking for an AI chatbot, an internal document search assistant, and an analytics dashboard powered by machine learning. We'd like to kick things off within three months."

print(prompt_passing(prompt).output_text)

{
  "client_name": "Jason",
  "company": "GreenTech Energy",
  "industry": "",
  "requirement": "AI chatbot, an internal document search assistant, and an analytics dashboard powered by machine learning",
  "urgency": "within three months",
  "missing_information": [
    "industry"
  ]
}


In [13]:
result = interaction.output_text

In [14]:
type(result)

str

In [15]:
import json

data = json.loads(result)

print(data)

{'client_name': 'Jason', 'company': 'GreenTech Energy', 'industry': '', 'requirement': 'AI chatbot, an internal document search assistant, and an analytics dashboard powered by machine learning', 'urgency': 'within three months', 'missing_information': ['industry']}


In [20]:
import json

with open("test_inputs.json", "r", encoding="utf-8") as file:
    enquiries = json.load(file)

for enquiry in enquiries:
    print(enquiry["id"])
    print(enquiry["category"])
    print(enquiry["client_enquiry"])
    print(prompt_passing(enquiry["client_enquiry"]).output_text)
    print("-" * 50)

1
Complete Enquiry
Hello, my name is Sarah Johnson, and I work at BrightMart Retail. We are looking for an AI chatbot that can answer customer queries on our website and integrate with Shopify. We'd like to launch it within the next two weeks for our marketing campaign.
{
  "client_name": "Sarah Johnson",
  "company": "BrightMart Retail",
  "industry": "Retail",
  "requirement": "AI chatbot that can answer customer queries on our website and integrate with Shopify",
  "urgency": "within the next two weeks",
  "missing_information": []
}
--------------------------------------------------
2
Complete Enquiry
Hi, I'm David Wilson from Nova Manufacturing. We want to develop an AI-powered predictive maintenance system that can analyze machine sensor data and alert our maintenance team before equipment fails. We hope to begin the project next month.
{
  "client_name": "David Wilson",
  "company": "Nova Manufacturing",
  "industry": "",
  "requirement": "develop an AI-powered predictive mainte

RateLimitError: Error code: 429 - {'error': {'message': 'You do not have enough quota to make this request.', 'code': 'too_many_requests'}}